In [ ]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
from openpiv import filters, pyprocess, validation

# --- PARAMETRI ---
INPUT_DIR = Path("F:\\Luca_Data\\20260925_test_microscope_and_camera\\fluoromax_0.25_1000fps_1uls")
OUTPUT_DIR = Path("results/piv")
PATTERN = "*.tif"  # Cambia l'estensione se le immagini non sono .tif

# Parametri PIV
WINDOW_SIZE = 32
OVERLAP = 16
SEARCH_AREA_SIZE = 64
DT_SECONDS = 1/1000        # 1000 fps (quindi 1 millisecondo tra i frame)
S2N_THRESHOLD = 1.3        # Soglia rapporto segnale/rumore

# Parametri di ritaglio (Crop)
# Ritaglia l'immagine per alleggerire l'analisi. Regola questi valori in base alla tua ROI.
CROP_Y_MIN = 100
CROP_Y_MAX = 800
CROP_X_MIN = 100
CROP_X_MAX = 900


In [ ]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
from openpiv import filters, pyprocess, validation

# --- PARAMETRI ---
INPUT_DIR = Path("F:\\Luca_Data\\20260925_test_microscope_and_camera\\fluoromax_0.25_1000fps_1uls")
OUTPUT_DIR = Path("results/piv")
PATTERN = "*.tif"  # Cambia l'estensione se le immagini non sono .tif

# Parametri PIV
WINDOW_SIZE = 32
OVERLAP = 16
SEARCH_AREA_SIZE = 64
DT_SECONDS = 1/1000        # 1000 fps (quindi 1 millisecondo tra i frame)
S2N_THRESHOLD = 1.3        # Soglia rapporto segnale/rumore

# Parametri di ritaglio (Crop)
# Ritaglia l'immagine per alleggerire l'analisi. Regola questi valori in base alla tua ROI.
CROP_Y_MIN = 100
CROP_Y_MAX = 800
CROP_X_MIN = 100
CROP_X_MAX = 900


In [ ]:

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def read_gray_and_crop(path):
    # Legge l'immagine in scala di grigi
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError(f"Impossibile leggere l'immagine: {path}")
    
    # Ritaglia l'immagine per isolare il canale/la zona di interesse
    cropped = image[CROP_Y_MIN:CROP_Y_MAX, CROP_X_MIN:CROP_X_MAX]
    return cropped

def analyze_pair(path_a, path_b):
    frame_a = read_gray_and_crop(path_a)
    frame_b = read_gray_and_crop(path_b)
    
    # Stima degli spostamenti con OpenPIV
    u, v, s2n = pyprocess.extended_search_area_piv(
        frame_a,
        frame_b,
        window_size=WINDOW_SIZE,
        overlap=OVERLAP,
        search_area_size=SEARCH_AREA_SIZE,
        dt=DT_SECONDS,
        sig2noise_method="peak2peak",
    )
    
    x, y = pyprocess.get_coordinates(
        image_size=frame_a.shape,
        search_area_size=SEARCH_AREA_SIZE,
        overlap=OVERLAP,
    )
    
    # NOTA: x e y sono relativi all'immagine ritagliata.
    # Se vuoi rimetterli alle coordinate originali, decommenta le righe sotto:
    # x = x + CROP_X_MIN
    # y = y + CROP_Y_MIN
    
    # Filtra i vettori poco affidabili
    # Un try/except per risolvere l'errore di openpiv (TypeError: sig2noise_val got multiple values for 'threshold')
    try:
        u, v, invalid = validation.sig2noise_val(u, v, s2n, threshold=S2N_THRESHOLD)
    except TypeError:
        u, v, invalid = validation.sig2noise_val(u, v, s2n, S2N_THRESHOLD)
        
    u, v = filters.replace_outliers(
        u, v, invalid,
        method="localmean",
        max_iter=3,
        kernel_size=3,
    )
    
    return x, y, u, v, s2n, invalid

def main():
    frames = sorted(INPUT_DIR.glob(PATTERN))
    if len(frames) < 2:
        raise RuntimeError(f"Servono almeno due immagini in {INPUT_DIR} con estensione {PATTERN}")
        
    print(f"Trovati {len(frames)} frame. Inizio l'analisi...")
    
    for index, (path_a, path_b) in enumerate(zip(frames, frames[1:])):
        x, y, u, v, s2n, invalid = analyze_pair(path_a, path_b)
        
        result = pd.DataFrame({
            "x_px": x.ravel(),
            "y_px": y.ravel(),
            "u_px_s": u.ravel(),
            "v_px_s": v.ravel(),
            "signal_to_noise": s2n.ravel(),
            "invalid": invalid.ravel(),
        })
        
        output_path = OUTPUT_DIR / f"piv_{index:04d}.csv"
        result.to_csv(output_path, index=False)
        
        # Stampa l'avanzamento ogni 10 frame per evitare di intasare l'output
        if index % 10 == 0:
            print(f"Analizzato e salvato: frame {index}/{len(frames)-1} -> {output_path.name}")
            
    print("\nAnalisi completata con successo!")

if __name__ == "__main__":
    main()
